In [1]:
import threading
import sys
import json
import copy
import os

import sqlite3
import numpy as np

In [2]:
sys.path.insert(0, "")

In [3]:
from GameExtractor import MlbGameExtractor
from BatterExtractor import BatterExtractor
from BatterSql import BatterSql
from PitcherSql import PitcherSql
from PlaySql import PlaySql
from GameSql import GameSql
from BioSql import BioSql
from TeamSql import TeamSql
from FielderSql import FielderSql
from PitchSql import PitchSql

In [4]:
statDbPath =  "./updated_database/baseball_info.db"
chadwickDbPath = "./updated_database/chadwick.db"

year = 2024

In [ ]:
years = [2015]#2025,2024,2023,2022,2021,2020,2019,2018,2017,2016]

for year in years:
    start_path = f"C:/Users/graye/dev/python/fantasyBaseballPlayerData/mlb_api_data/{year}"
    
    counter = 0
    
    #MAYBE JUST GET A LIST OF BATTER/PITCHERIDS HERE AND UPDATE THOSE ONLY, THEN WE MINIMIZE THE WORK NEEDED
    for root, dirs, files in os.walk(start_path):
        for file in files:
            file_path = os.path.join(root, file)
    
            with open(file_path, 'r') as file:
                gameJson      = json.load(file)
                gameExtractor = MlbGameExtractor(gameJson)
    
                try:
                    game     = gameExtractor.getGameFromJson()
                    fielders = gameExtractor.getFielders()
                    batters  = gameExtractor.getBatters     ()
                    pitchers = gameExtractor.getPitchers    ()
                    plays,pitches = gameExtractor.getPlaysAndPitches()
                
                    games = []
        
                    games.append(game)
        
                    GameSql    .insertGames   (games   , statDbPath)
                    BatterSql  .insertBatters (batters , statDbPath)
                    PitcherSql .insertPitchers(pitchers, statDbPath)
                    PlaySql    .insertPlays   (plays   , statDbPath)
                    PitchSql.insertPitches(pitches, statDbPath)
                    FielderSql.insertFielders(fielders, statDbPath)
                    
                except Exception as e:
                    print(file_path)
                    print(game.gid)
                    # for fielder in fielders:
                    #     print(fielder.fid)
                    print(e)
    
                    raise
    
                counter = counter + 1
    
                if counter % 500 == 0:
                    print(counter)

In [ ]:
print(year)

In [5]:
years = [2024,2023,2022,2021,2020,2019,2018,2017,2016,2015]

for year in years:
    latestHitters = BatterSql.getHittersFromSeason(year, statDbPath)

    print(year)
    print(len(latestHitters))
    
    count = 0
    
    for hitter in latestHitters:
        #BatterSql.updateSeasonStatsHitting  (hitter, year,           statDbPath)
        #BatterSql.updateLifeTimeStatsHitting(hitter,                 statDbPath)
        #BioSql.   updateBio                 (hitter, chadwickDbPath, statDbPath)
        #BatterSql.updateSeasonBabip      (hitter, year, statDbPath)
        #BatterSql.updateRunsPerTob(hitter, year, statDbPath)
        #BatterSql.updateRbisPerBip(hitter, year, statDbPath)
        #BatterSql.updateSeasonSpd(hitter, year, statDbPath)
        BatterSql.updateSeasonTrajectoryPercentages(hitter, year, statDbPath)
    
        count += 1

        if count % 100 == 0:
            print(count)
    
    #TeamSql.updateTeamStatsHitting(statDbPath, year)
    #
    #BatterSql.updateSeasonGrades     (year, statDbPath)
    #BatterSql.updateSeasonTotalGrades(year, statDbPath)

2024
741
100
200
300
400
500
600
700
2023
765
100
200
300
400
500
600
700
2022
790
100
200
300
400
500
600
700
2021
1373
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
2020
618
100
200
300
400
500
600
2019
1287
100
200
300
400
500
600
700
800
900
1000
1100
1200
2018
1270
100
200
300
400
500
600
700
800
900
1000
1100
1200
2017
1229
100
200
300
400
500
600
700
800
900
1000
1100
1200
2016
1247
100
200
300
400
500
600
700
800
900
1000
1100
1200
2015
1252
100
200
300
400
500
600
700
800
900
1000
1100
1200


In [ ]:
latestPitchers = PitcherSql.getPitchersFromSeason(year, statDbPath)

for pitcher in latestPitchers:
    PitcherSql.updateSeasonStatsPitching  (pitcher, year,           statDbPath)
    PitcherSql.updateLifetimeStatsPitching(pitcher,                 statDbPath)
    BioSql    .updateBio                  (pitcher, chadwickDbPath, statDbPath)

PitcherSql.updateSeasonGrades(year, statDbPath)

TeamSql.updateTeamStatsPitching(statDbPath, year)

In [ ]:
latestFielders = FielderSql.getFieldersFromSeason(year, statDbPath)

for fielder in latestFielders:
    FielderSql.updateSeasonStatsFielding(fielder, year, statDbPath)

In [ ]:
#Team grade
era -> runs, rbis
whip -> obp
stolen bases -> sb
home runs -> hr

obp -> whip
strikeouts -> strikeuts
runs -> era

#trends
#schedule
#matchup grades

In [ ]:
teamToHittingGradeAndPercentile = TeamSql.updateMatchupGradesHitting(statDbPath, 2025)

In [ ]:
for team in teamToHittingGradeAndPercentile:
    grade,percentile = teamToHittingGradeAndPercentile[team]

    print(f"{team}: {grade}, {percentile}")

In [ ]:
teamToPitchingGradeAndPercentile = TeamSql.updateMatchupGradesPitching(statDbPath, 2025)

In [ ]:
for team in teamToPitchingGradeAndPercentile:
    grade,percentile = teamToPitchingGradeAndPercentile[team]

    print(f"{team}: {grade}, {percentile}")

In [ ]:
#(Hits - Home Runs) / (At-Bats - Strikeouts - Home Runs + Sacrifice Flies)

babipStatsQuery = "SELECT hits,homeRuns,atBats,strikeOuts,sacFlies FROM SeasonStatsHitting WHERE playerId=\"677594\" and season=2022"

connection = sqlite3.connect(statDbPath)
cursor     = connection.cursor()

cursor.execute(babipStatsQuery)

rows = cursor.fetchall()

if len(rows) != 1:
    print("oops")

else:
    row = rows[0]
    
    hits       = float(row[0])
    homeRuns   = float(row[1])
    atBats     = float(row[2])
    strikeOuts = float(row[3])
    sacFlies   = float(row[4])
    
    babip = (hits - homeRuns) / (atBats - strikeOuts - homeRuns + sacFlies)
    
    print(babip)

cursor.close()
connection.close()

In [ ]:
atBatPerExtraBaseHitQuery = "SELECT atBats,doubles,triples FROM SeasonStatsHitting WHERE playerId=\"677594\" and season=2024"

connection = sqlite3.connect(statDbPath)
cursor     = connection.cursor()

cursor.execute(atBatPerExtraBaseHitQuery)

rows = cursor.fetchall()

if len(rows) != 1:
    print("oops")

else:
    row = rows[0]

    atBats  = float(row[0])
    doubles = float(row[1])
    triples = float(row[2])

    atBatPerDouble = atBats / doubles
    atBatPerTriple = float('inf') if triples == 0 else atBats / triples

    print(atBatPerDouble)
    print(atBatPerTriple)

cursor.close()
connection.close()

In [ ]:
atBatPerExtraBaseHitQuery = "SELECT plateAppearances,hitByPitch FROM SeasonStatsHitting WHERE playerId=\"677594\" and season=2024"

connection = sqlite3.connect(statDbPath)
cursor     = connection.cursor()

cursor.execute(atBatPerExtraBaseHitQuery)

rows = cursor.fetchall()

if len(rows) != 1:
    print("oops")

else:
    row = rows[0]

    pa  = float(row[0])
    hbp = float(row[1])

    paPerHbp = float('inf') if hbp == 0 else pa / hbp

    print(paPerHbp)

cursor.close()
connection.close()

In [ ]:
atBatPerExtraBaseHitQuery = "SELECT plateAppearances,sacFlies FROM SeasonStatsHitting WHERE playerId=\"677594\" and season=2024"

connection = sqlite3.connect(statDbPath)
cursor     = connection.cursor()

cursor.execute(atBatPerExtraBaseHitQuery)

rows = cursor.fetchall()

if len(rows) != 1:
    print("oops")

else:
    row = rows[0]

    pa  = float(row[0])
    sf  = float(row[1])

    paPerSf = float('inf') if sf == 0 else pa / sf

    print(paPerSf)

cursor.close()
connection.close()

In [ ]:
#HR/FB%, LD%, GB&, FB%, SPD, swing/contact %, IFBB%, true FB%, Hard%, Oppo%

In [ ]:
stolenBasesStatsQuery = "SELECT stolenBases,caughtStealing,hits,walks,hitByPitch FROM SeasonStatsHitting WHERE playerId=\"677594\" and season=2024"

connection = sqlite3.connect(statDbPath)
cursor     = connection.cursor()

cursor.execute(stolenBasesStatsQuery)

rows = cursor.fetchall()

if len(rows) != 1:
    print("oops")

else:
    row = rows[0]

    sb    = float(row[0])
    cs    = float(row[1])
    hits  = float(row[2])
    walks = float(row[3])
    hbp   = float(row[4])

    sbaPerTobDenom    = hits + walks + hbp
    sbPercentageDenom = sb + cs

    sbaPerTob    = float('inf') if sbaPerTobDenom == 0.0 else (sb + cs) / (hits + walks + hbp)
    sbPercentage = float('inf') if sbPercentageDenom == 0.0 else (sb) / (sbPercentageDenom)

    print(sbaPerTob)
    print(sbPercentage)

cursor.close()
connection.close()

In [ ]:
query = "SELECT plateAppearances,walks,intentionalWalks,strikeOuts FROM SeasonStatsHitting WHERE playerId=\"677594\" and season=2024"

connection = sqlite3.connect(statDbPath)
cursor     = connection.cursor()

cursor.execute(query)

rows = cursor.fetchall()

if len(rows) != 1:
    print("oops")

else:
    row = rows[0]

    pa    = float(row[0])
    walks = float(row[1])
    iws   = float(row[2])
    sos   = float(row[3])

    walkPercentage = float('inf') if pa == 0.0 else walks / pa
    iwPercentage   = float('inf') if pa == 0.0 else iws / pa
    soPercentage   = float('inf') if pa == 0.0 else sos / pa

    print(walkPercentage)
    print(iwPercentage)
    print(soPercentage)

cursor.close()
connection.close()

In [ ]:
#runs/tob, rbi/bip

query = "SELECT runs,homeRuns,hits,walks,hitByPitch,rbis,sacFlies,atBats,strikeOuts FROM SeasonStatsHitting WHERE playerId=\"677594\" and season=2024"

connection = sqlite3.connect(statDbPath)
cursor     = connection.cursor()

cursor.execute(query)

rows = cursor.fetchall()

if len(rows) != 1:
    print("oops")

else:
    row = rows[0]

    runs       = float(row[0])
    homeRuns   = float(row[1])
    hits       = float(row[2])
    walks      = float(row[3])
    hitByPitch = float(row[4])
    rbis       = float(row[5])
    sacFlies   = float(row[6])
    atBats     = float(row[7])
    strikeOuts = float(row[8])

    runsPerTobDenom = hits + walks + hitByPitch - homeRuns
    rbisPerBipDenom = atBats - homeRuns - strikeOuts

    runsPerTobDenom = float('inf') if runsPerTobDenom == 0.0 else (runs - homeRuns) / runsPerTobDenom
    rbisPerBip      = float('inf') if rbisPerBipDenom == 0.0 else (rbis - (homeRuns * 1.565) - sacFlies) / rbisPerBipDenom

    print(runsPerTobDenom)
    print(rbisPerBip)

cursor.close()
connection.close()

In [ ]:
SELECT COUNT(*) FROM Pitches INNER JOIN Plays ON Pitches.play = Plays.key INNER JOIN Games ON Games.gameId = Plays.gameId WHERE Pitches.hitTrajectory NOT NULL AND Pitches.inPlay="true" AND Plays.batterId="677594" AND Games.season=2024